In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 1.5,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/09 22:42:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/09 22:42:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/09 22:42:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


New Spark session created successfully


25/04/09 22:42:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
# adding dependencies to spark's context so spark workers (the threads) can access them

In [8]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [9]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [10]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py                         (0 + 12) / 12]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
processSub sub-004
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
processSub sub-023
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_pat

Processed 2247790 records for Alzheimer's group
Processed 1868405 records for Control group
CPU times: user 114 ms, sys: 118 ms, total: 232 ms
Wall time: 4min 33s


In [11]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df
cntrl_df = group_c_spark_df

In [12]:
alz_df.show()

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|  7.020328E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|  3.399499E-4|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power|  0.087231696|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power| 0.0011391409|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy|   0.34805238| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower|  0.011235955| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power| 0.0014689578|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|  5.043481E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power|   0.08462298|      band|
|  sub-008|   ep-0|      Fp2|   Theta|      Power|  0.002023743|      band|
|  sub-008| 

In [13]:
cntrl_df.head()

Row(SubjectID='sub-037', EpochID='ep-0', Electrode='Fp1', WaveBand='Alpha', FeatureName='Power', FeatureValue=0.004162836819887161, table_type='band')

In [14]:
NUM_TEST_SUBJECTS_PER_GROUP = 2  # i know before we had three , but 2 is better 3 took out too much data.

alz_test_subjects = (
    alz_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


cntrl_test_subjects = (
    cntrl_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


In [15]:
print(f"azl test subjects {alz_test_subjects}\ncntrl test subjects {cntrl_test_subjects}")

azl test subjects ['sub-001', 'sub-002']
cntrl test subjects ['sub-037', 'sub-038']


In [29]:
from pyspark.sql.functions import lit

In [30]:
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [22]:
alz_df.show(2)
cntrl_df.show(2)

+---------+-------+---------+--------+-----------+------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+------------+----------+-----+
|  sub-008| ep-476|       P3|   Theta|      Power|0.0023075177|      band|    1|
|  sub-008| ep-496|       O2|    NULL| TotalPower| 0.011235955| electrode|    1|
+---------+-------+---------+--------+-----------+------------+----------+-----+
only showing top 2 rows

+---------+-------+---------+--------+-----------+------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+------------+----------+-----+
|  sub-041| ep-337|       T4|   Delta|      Power|  0.08452765|      band|    0|
|  sub-045|  ep-59|       F3|   Alpha|      Power|0.0031511658|      band|    0|
+---------+-------+---------+--------+-----------+------------+----------+-----+
onl

In [23]:
# Filter test rows
alz_test_df = alz_df.filter(alz_df.SubjectID.isin(alz_test_subjects))
cntrl_test_df = cntrl_df.filter(cntrl_df.SubjectID.isin(cntrl_test_subjects))

# Filter training rows (not in test subjects)
alz_train_df = alz_df.filter(~alz_df.SubjectID.isin(alz_test_subjects))
cntrl_train_df = cntrl_df.filter(~cntrl_df.SubjectID.isin(cntrl_test_subjects))

In [25]:
# Individual counts
alz_train_count = alz_train_df.select("SubjectID", "EpochID").distinct().count()
alz_test_count = alz_test_df.select("SubjectID", "EpochID").distinct().count()
cntrl_train_count = cntrl_train_df.select("SubjectID", "EpochID").distinct().count()
cntrl_test_count = cntrl_test_df.select("SubjectID", "EpochID").distinct().count()

# Combined counts
test_total_count = alz_test_df.unionByName(cntrl_test_df).select("SubjectID", "EpochID").distinct().count()
train_total_count = alz_train_df.unionByName(cntrl_train_df).select("SubjectID", "EpochID").distinct().count()

# Print them out
print(f"Alzheimer's train: {alz_train_count}")
print(f"Alzheimer's test:  {alz_test_count}")
print(f"Control train:     {cntrl_train_count}")
print(f"Control test:      {cntrl_test_count}")
print(f"Total test:        {test_total_count}")
print(f"Total train:       {train_total_count}")

Alzheimer's train: 18621
Alzheimer's test:  925
Control train:     15135
Control test:      1112
Total test:        2037
Total train:       33756


In [26]:
train_df = alz_train_df.unionByName(cntrl_train_df)
test_df = alz_test_df.unionByName(cntrl_test_df)

 # Now need to pivot and create the 3 tables, (band specific table, electrode specific table and epoch specific table

In [28]:
sub1.filter((sub1.table_type=="band"))
sub1.filter((sub1.table_type=="electrode"))
sub1.filter((sub1.table_type=="epoch"))

band_df = band_df.withColumn("Electrode_WaveBand", concat_ws("_", "Electrode", "WaveBand"))

band_pivot = band_df.groupBy("SubjectID", "EpochID").pivot("Electrode_WaveBand").agg(first("FeatureValue"))


NameError: name 'band_df' is not defined

In [27]:
train_df.show()

+---------+-------+---------+--------+-----------+------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+------------+----------+-----+
|  sub-008| ep-476|       P3|   Theta|      Power|0.0023075177|      band|    1|
|  sub-008| ep-496|       O2|    NULL| TotalPower| 0.011235955| electrode|    1|
|  sub-008| ep-138|       P3|   Theta|      Power| 0.003857784|      band|    1|
|  sub-017| ep-488|       T5|   Theta|      Power|0.0052539054|      band|    1|
|  sub-008| ep-322|       F3|   Delta|      Power| 0.085532315|      band|    1|
|  sub-008| ep-408|       O1|    NULL| TotalPower| 0.011235955| electrode|    1|
|  sub-025|  ep-67|       Fz|   Alpha|      Power| 2.621561E-4|      band|    1|
|  sub-025|  ep-51|       F7|   Delta|      Power| 0.084310316|      band|    1|
|  sub-017| ep-437|       P3|    NULL|TotalEnergy|  0.45740035| electrode|    1|
|  sub-008| ep-491|       P3

In [ ]:
from dimensionality_reduction import normalize_power # it z-scores the data
# NOTE, this uses the first parameters for mean and std for the z-score, so none of test_df's data is used to z-score
train_df, test_df = normalize_power(train_df, test_df) 

In [ ]:
from pyspark.sql.functions import col

# Filter cases where WaveBand is 'Total' and Power is not zero
non_zero_total_power = train_df.filter(
    (col("WaveBand") == "Total") & (col("Power") != 0)
)

# Count and show a few examples
count = non_zero_total_power.count()
print(f"Number of rows where WaveBand = 'Total' and Power != 0: {count}")

if count > 0:
    non_zero_total_power.show(5)


In [ ]:
from dimensionality_reduction import prepare_features_for_pca
# this pivots the tables so that its better suited for PCA and ML with pyspark's libraries
train_df, train_features_column = prepare_features_for_pca(train_df)
test_df, test_features_column = prepare_features_for_pca(test_df)

In [ ]:
train_features_column.sort()
test_features_column.sort()
if train_features_column != test_features_column:
    print("!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!")

In [ ]:
from dimensionality_reduction import fit_pca_model
#Note , this finds the features to explain the model's PCA
K_VAR_TARGET=0.95
pca_model_func, k_val = fit_pca_model(train_df, train_features_column, variance_target=K_VAR_TARGET)

In [ ]:
print(f"We can explian {K_VAR_TARGET} with {k_val} features. That is a lot less then {len(train_df.columns)-3} features we originally had (we hope).") #-3 for subjectID , epochID and lebel

In [ ]:
# know that we know we can explain 95% of the variance (or what we set target to) ,
# lets make our dataframces only have those important columns
from dimensionality_reduction import apply_pca_model
train_df = apply_pca_model(train_df, train_features_column, pca_model_func, k_val)
test_df  = apply_pca_model(test_df, train_features_column, pca_model_func, k_val)

In [ ]:
train_df.printSchema()

In [ ]:
train_df.head()

# ML

In [ ]:
model_summaries = []

In [ ]:
%%time
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Get input size from PCA features

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=[k_val, 100, 2],  # input → hidden (100 units) → 2 output classes
    maxIter=1000,
    seed=42
)

mlp_model = mlp.fit(train_df)
mlp_preds = mlp_model.transform(test_df)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mlp_auc = evaluator.evaluate(mlp_preds)

print("MLP AUC:", mlp_auc)

In [ ]:
preds_pd = mlp_preds.select("prediction", "label").toPandas()

from sklearn.metrics import classification_report, accuracy_score

print("Neural Network accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

In [ ]:
from sklearn.metrics import classification_report

report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

model_summaries.append({
    "model": "Neural Network",
    "auc": mlp_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=100)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
gbt_auc = evaluator.evaluate(gbt_preds)
print("Gradient Boosted Trees AUC:", gbt_auc)

preds_pd = gbt_preds.select("prediction", "label").toPandas()
print("GBT accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

In [ ]:
# Generate classification report for GBT
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append GBT results to summary
model_summaries.append({
    "model": "Gradient Boosted Trees",
    "auc": gbt_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
tree_model = tree.fit(train_df)
tree_preds = tree_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
tree_auc = evaluator.evaluate(tree_preds)
print("Decision Tree AUC:", tree_auc)

preds_pd = tree_preds.select("prediction", "label").toPandas()
print("Decision Tree accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


In [ ]:
# Generate classification report for Decision Tree
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append Decision Tree results to summary
model_summaries.append({
    "model": "Decision Tree",
    "auc": tree_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
from pyspark.ml.classification import LinearSVC

# Train SVM model
svm = LinearSVC(featuresCol="features", labelCol="label", maxIter=100, regParam=0.1)
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)

# Evaluate SVM
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
svm_auc = evaluator.evaluate(svm_preds)
print("SVM AUC:", svm_auc)

# Accuracy and report
preds_pd = svm_preds.select("prediction", "label").toPandas()
from sklearn.metrics import classification_report, accuracy_score

print("SVM accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


In [ ]:
# Generate classification report for SVM
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append SVM results to summary
model_summaries.append({
    "model": "SVM",
    "auc": svm_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col

# Step 1: Fit LSH model on training data
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=0.25,
    numHashTables=6
)
lsh_model = lsh.fit(train_df)

# Step 2: Perform approximate similarity join between test and train
# This will find the approximate nearest neighbors of test samples in train set
similarities = lsh_model.approxSimilarityJoin(
    datasetA=test_df,
    datasetB=train_df,
    threshold=float("inf"),  # You can limit this if needed
    distCol="euclidean_distance"
)

# Step 3: For each test point, pick nearest neighbor (smallest distance)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("datasetA").orderBy("euclidean_distance")

nearest_neighbors = similarities \
    .withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1)

# Step 4: Collect prediction from nearest training label
predictions = nearest_neighbors.select(
    col("datasetA.label").alias("true_label"),
    col("datasetB.label").alias("predicted_label")
)

# Step 5: Evaluate
preds_pd = predictions.toPandas()

from sklearn.metrics import accuracy_score, classification_report

print("Approximate KNN accuracy:", accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))


In [ ]:
# Ensure columns are correctly named
if "true_label" not in preds_pd.columns:
    preds_pd.columns = ["true_label", "predicted_label"]

# Generate classification report
report_str = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=True)

# Append Approximate KNN results to summary
model_summaries.append({
    "model": "Approximate KNN",
    "auc": "N/A",
    "accuracy": accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
print("\n\n=== MODEL PERFORMANCE SUMMARY ===")
for summary in model_summaries:
    print(f"\nModel: {summary['model']}")
    
    auc = summary.get('auc', 'N/A')
    if isinstance(auc, (int, float)):
        print(f"  AUC:      {auc:.4f}")
    else:
        print(f"  AUC:      {auc}")

    acc = summary.get('accuracy', 'N/A')
    if isinstance(acc, (int, float)):
        print(f"  Accuracy: {acc:.4f}")
    else:
        print(f"  Accuracy: {acc}")
    
    print("\n  Classification Report:")
    print(summary.get("report_str", "  No report available"))

print("\n\n=== ACTIVE CONFIGURATION ===")
from pprint import pprint
pprint(load_config())
from datetime import datetime

now = datetime.now()
print("Current date and time:", now)

In [ ]:

import sys
from datetime import datetime

# Open the log file in append mode
logfile = open("model_results_log.txt", "a")

# Define a dual-output print function
def log_print(*args, **kwargs):
    print(*args, **kwargs)
    print(*args, **kwargs, file=logfile)

# === Your original code, now using `log_print` ===
log_print("\n\n=== MODEL PERFORMANCE SUMMARY ===")
for summary in model_summaries:
    log_print(f"\nModel: {summary['model']}")
    
    auc = summary.get('auc', 'N/A')
    if isinstance(auc, (int, float)):
        log_print(f"  AUC:      {auc:.4f}")
    else:
        log_print(f"  AUC:      {auc}")

    acc = summary.get('accuracy', 'N/A')
    if isinstance(acc, (int, float)):
        log_print(f"  Accuracy: {acc:.4f}")
    else:
        log_print(f"  Accuracy: {acc}")
    
    log_print("\n  Classification Report:")
    log_print(summary.get("report_str", "  No report available"))

log_print("\n\n=== ACTIVE CONFIGURATION ===")
from pprint import pprint
from io import StringIO

# Capture pprint output
config_str = StringIO()
pprint(load_config(), stream=config_str)
log_print(config_str.getvalue())

# Add date and time
now = datetime.now()
log_print("Current date and time:", now)

logfile.close()  # Always close the file when done
